# 0.5) Sub-grid relief of the EC-Earth3 cells (ETOPO1)

**Why.** The BIO variables are means over a 0.7° cell (≈ 78 km). In a mountain cell the crop grows in the valleys,
warmer than the cell mean, so a cold cell mean can carry real presences (3.h: 0.4–3.7 % of the presence cells are
below the FAO ECOCROP TMIN, Andes, Himalaya, south-west China). The SVM then learns that some cold climates hold the
crop, and gives the flat boreal cells of the same mean temperature an intermediate probability (the boreal tail of
3.h and 5.h). A measure of the **heterogeneity of the cell** tells the two apart: cold and heterogeneous = warm
valleys possible; cold and flat = no warmer place in the cell.

**Variable `RELIEF`** = standard deviation of the elevation of the land pixels of the cell (m), the usual measure of
topographic heterogeneity. Independent of the absolute elevation (a flat plateau, the Antarctic ice sheet or the
taiga give ≈ 0; a cell of valleys and peaks several hundred metres). × 6.5 °C km⁻¹ (standard lapse rate) it reads as
the typical temperature spread inside the cell. Also saved: mean, 10th and 90th percentiles of the elevation.
One static layer: the same for the five scenarios (the delta method keeps the sub-grid structure) and for every
crop; computed on all land, with no land use. It is a proxy of the sub-grid climate, not an agronomic slope
constraint (in GAEZ steep slopes reduce suitability; here relief can go with presence in cold cells).

**Data**: ETOPO1 1 arc-min, ice surface (top of the ice sheets, the surface the climate sees), cell-registered
(NOAA; BDAP `base/Elevation/GLOBAL/ETOPO1`). Land pixels = Natural Earth 1:10m land at 5 arc-min, as the land mask
of 0.1; each 1′ pixel goes to the EC-Earth3 cell that contains its centre.

**Check before test 6**: for the cells below TMIN, is the relief of the presences higher than that of the cropland
absences and of the flat land north of 50° N outside the cropland? (AUC of the relief alone.)

Output: `<out_root>/0_masks_ecearth3/relief_ecearth3.nc` (`RELIEF`, `elev_mean`, `elev_p10`, `elev_p90`, `n_pix`).

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import time
import numpy as np, pandas as pd, xarray as xr, rasterio, regionmask
from rasterio.windows import Window
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from pathlib import Path
from sklearn.metrics import roc_auc_score
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())  # racine du projet (local ou BDAP)
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu   # helpers + config reader (project root)
CFG = cu.load_config()   # config/config_<profile>.json, written by 00_config_and_inventory.ipynb

DEM = cu.data_file(CFG, "etopo1")
assert DEM.exists(), f"ETOPO1 not found: {DEM} (run 00_config_and_inventory.ipynb after adding etopo1)"
OUT = cu.out_dir(CFG, cu.MASKS)
GRID = cu.svm_grid(CFG)
NLAT, NLON = GRID["lat"].size, GRID["lon"].size
LAND = cu.land_mask(CFG).values
with rasterio.open(DEM) as src:
    H, W, T, NODATA = src.height, src.width, src.transform, src.nodata
lat1 = T.f + T.e * (np.arange(H) + 0.5)                  # pixel centres (north -> south)
lon1 = T.c + T.a * (np.arange(W) + 0.5)
I1, _ = cu.cell_of(lat1, np.zeros(H), GRID)              # EC-Earth3 row of each DEM row
_, J1 = cu.cell_of(np.zeros(W), lon1, GRID)              # EC-Earth3 column of each DEM column
lat5 = 90 - (np.arange(2160) + 0.5) / 12; lon5 = -180 + (np.arange(4320) + 0.5) / 12
land5 = regionmask.defined_regions.natural_earth_v5_0_0.land_10.mask(lon5, lat5).notnull().values   # as 0.1
R5 = np.clip(np.floor((90 - lat1) * 12).astype(int), 0, 2159)   # 5' land pixel of each DEM pixel
C5 = np.clip(np.floor((lon1 + 180) * 12).astype(int), 0, 4319)
print(f"{DEM.name}: {H} x {W}, {abs(T.a) * 60:.2f} arc-min | outputs {OUT}")

## 1. Elevation statistics per EC-Earth3 cell, one EC-Earth3 row at a time

In [ ]:
stat = {k: np.full(NLAT * NLON, np.nan) for k in ("RELIEF", "elev_mean", "elev_p10", "elev_p90", "n_pix")}
t0 = time.time()
with rasterio.open(DEM) as src:
    for i in range(NLAT):
        rows = np.where(I1 == i)[0]
        if rows.size == 0:
            continue
        r0, r1 = rows.min(), rows.max() + 1               # contiguous: latitude is monotonic
        z = src.read(1, window=Window(0, r0, W, r1 - r0)).astype(np.float64)
        land = land5[R5[r0:r1]][:, C5]
        if NODATA is not None:
            land &= z != NODATA
        jj, zz = np.broadcast_to(J1, z.shape)[land], z[land]
        if zz.size == 0:
            continue
        o = np.lexsort((zz, jj)); jj, zz = jj[o], zz[o]
        u, start, n = np.unique(jj, return_index=True, return_counts=True)
        m = np.add.reduceat(zz, start) / n
        k = i * NLON + u
        stat["elev_mean"][k] = m
        stat["RELIEF"][k] = np.sqrt(np.maximum(np.add.reduceat(zz ** 2, start) / n - m ** 2, 0.0))
        stat["elev_p10"][k] = zz[start + ((n - 1) * 0.1).astype(int)]
        stat["elev_p90"][k] = zz[start + ((n - 1) * 0.9).astype(int)]
        stat["n_pix"][k] = n
        if i % 32 == 0:
            print(f"  row {i}/{NLAT} ({GRID['lat'][i]:.0f} deg) ... {time.time() - t0:.0f} s", flush=True)
ATTRS = {"RELIEF": {"long_name": "sub-grid relief: standard deviation of the elevation of the land pixels of the cell", "units": "m"},
         "elev_mean": {"long_name": "mean elevation of the land pixels", "units": "m"},
         "elev_p10": {"long_name": "10th percentile of the elevation of the land pixels", "units": "m"},
         "elev_p90": {"long_name": "90th percentile of the elevation of the land pixels", "units": "m"},
         "n_pix": {"long_name": "number of 1-arc-min land pixels in the cell"}}
REL = xr.Dataset({k: (("lat", "lon"), np.where(LAND, v.reshape(NLAT, NLON), np.nan).astype("float32"), ATTRS[k])
                  for k, v in stat.items()}, coords={"lat": GRID["lat"], "lon": GRID["lon"]},
                 attrs={"source": f"{DEM.name} (ETOPO1, NOAA), ice surface; land = Natural Earth 1:10m at 5 arc-min (as 0.1)",
                        "note": "static: the same for every scenario and crop"})
REL.to_netcdf(OUT / "relief_ecearth3.nc")
print(f"done in {time.time() - t0:.0f} s; land cells without DEM pixel: {int((LAND & ~np.isfinite(REL.RELIEF.values)).sum())}")

## 2. Does the relief separate the cold presences from the cold flat land?

For each crop, the cells below its ECOCROP TMIN (ERA5, 1.1.d): the training presences, the training absences
(cropland without the crop) and the land north of 50° N outside the cropland. AUC of `RELIEF` alone: 0.5 = no
separation, 1 = the cold presences always more heterogeneous.

In [ ]:
CHOSEN_VARIANT, CELL_AREA_THR = "A_60d", 0.0005                 # as 3.g / 4.g
ERA5_NAME, ERA5_PERIOD = "ERA5", ("1981", "2014")
CROPS = ["Maize_rf", "Potato_rf", "Soybean_rf", "Rice_rf", "Wheat_rf", "Wheat-winter_rf", "Wheat-spring_rf"]
ECOCROP = {"Maize_rf": (10, "season"), "Potato_rf": (7, "season"), "Soybean_rf": (10, "season"), "Rice_rf": (10, "season"),
           "Wheat-spring_rf": (5, "season"), "Wheat-winter_rf": (5, "phase2"), "Wheat_rf": (5, "phase2")}   # as 3.h
TAG = f"{ERA5_NAME}_{ERA5_PERIOD[0]}_{ERA5_PERIOD[1]}_ecearth3"
ERA5_DIR = cu.out_dir(CFG, "1_biovariables", f"{ERA5_NAME}_{cu.route(CELL_AREA_THR)}", CHOSEN_VARIANT)
CRO_ANY = cu.mirca_cropland(CFG).values
LAT2 = np.broadcast_to(GRID["lat"][:, None], LAND.shape)
R = REL.RELIEF.values
rows = []
for crop in CROPS:
    tr = pd.read_csv(ERA5_DIR / f"training_dataset_{TAG}_{crop}.csv", usecols=["lat", "lon", "label"])
    ii = np.searchsorted(GRID["lat"], tr.lat.values - 1e-6); jj = np.searchsorted(GRID["lon"], tr.lon.values - 1e-6)
    with xr.open_dataset(ERA5_DIR / f"BIOvariables_{TAG}_{crop}.nc") as ds:
        tg = cu.growing_temperature(ds, ECOCROP[crop][1])
    cold = np.isfinite(tg) & (tg < ECOCROP[crop][0])
    c_tr, y, r_tr = cold[ii, jj], tr.label.values, R[ii, jj]
    groups = {"presences": r_tr[(y == 1) & c_tr], "cropland absences": r_tr[(y == 0) & c_tr],
              "north of 50 N, outside cropland": R[LAND & ~CRO_ANY & cold & (LAT2 > 50)],
              "all presences (warm and cold)": r_tr[y == 1]}
    r = {"crop": crop}
    for g, v in groups.items():
        v = v[np.isfinite(v)]
        r[f"{g}: cells"] = v.size
        r[f"{g}: median RELIEF (m)"] = np.median(v) if v.size else np.nan
    for g in ("cropland absences", "north of 50 N, outside cropland"):
        a, b = groups["presences"], groups[g]
        a, b = a[np.isfinite(a)], b[np.isfinite(b)]
        r[f"AUC RELIEF, cold presences vs {g}"] = (roc_auc_score(np.r_[np.ones(a.size), np.zeros(b.size)], np.r_[a, b])
                                                    if a.size and b.size else np.nan)
    rows.append(r)
chk = pd.DataFrame(rows).set_index("crop")
chk.to_csv(OUT / "relief_check_cold_cells.csv")
with pd.option_context("display.max_columns", 30, "display.width", 250):
    display(chk.round(2))

## 3. Map

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
im = ax.pcolormesh(GRID["lon"], GRID["lat"], np.clip(REL.RELIEF.values, 1, None), norm=LogNorm(1, 2000), cmap="terrain", shading="nearest")
plt.colorbar(im, ax=ax, label="RELIEF: std of the elevation in the cell (m)")
ax.set_title("Sub-grid relief of the EC-Earth3 cells (ETOPO1)")
plt.savefig(OUT / "relief_ecearth3.png", dpi=90, bbox_inches="tight"); plt.show()